# Inter-Subject Correlation on a Naturalistic Movie

When people watch the **same movie**, their brains respond in similar,
time-locked ways. **Inter-subject correlation (ISC)** measures that shared
response. We'll learn it on simulated data, then apply the *same code* to the
real **Partly Cloudy** movie-fMRI dataset.

Part 1 needs only `numpy`, `scipy`, `matplotlib`. Part 2 adds `nilearn`.
Full write-up in `README.md`.

## Part 1 - ISC mechanics (simulated data)

Model each subject's signal in a region as shared + private:
$$x_i(t) = \sqrt{c}\,s(t) + \sqrt{1-c}\,e_i(t)$$
`c` is the fraction shared across people; larger `c` -> higher ISC. We make an
**auditory** region (c=0.5), a **social/STS** region (c=0.3), and a **control**
region (c=0, a clean null).

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

REGIONS = {'auditory': 0.50, 'social/STS': 0.30, 'control': 0.00}

def simulate(n_subjects=20, n_timepoints=300, regions=REGIONS, seed=0):
    rng = np.random.default_rng(seed)
    data = {}
    for name, c in regions.items():
        shared = rng.standard_normal(n_timepoints)
        private = rng.standard_normal((n_subjects, n_timepoints))
        data[name] = np.sqrt(c)*shared + np.sqrt(1-c)*private
    return data

data = simulate()
print({k: v.shape for k, v in data.items()})   # (subjects, timepoints)

### Leave-one-out ISC
Correlate **each subject** with the **average of the others**.

In [ ]:
def leave_one_out_isc(ts):                 # ts: (subjects, timepoints)
    n = ts.shape[0]; out = np.empty(n)
    for i in range(n):
        m = np.delete(ts, i, axis=0).mean(axis=0)
        a, b = ts[i]-ts[i].mean(), m-m.mean()
        out[i] = (a @ b) / np.sqrt((a @ a)*(b @ b))
    return out

def fisher_mean(r):                         # average correlations properly
    return float(np.tanh(np.mean(np.arctanh(np.clip(r, -0.999, 0.999)))))

for name, ts in data.items():
    iscs = leave_one_out_isc(ts)
    t, p = stats.ttest_1samp(iscs, 0.0)
    print(f'{name:10s} ISC = {fisher_mean(iscs):.3f}   (t={t:5.1f}, p={p:.1e})')

### Is it real? A circular-shift null
Shifting each subject's time course by a random amount keeps their own
temporal structure but destroys cross-subject alignment, so true ISC should
collapse to ~0. We build a null distribution and test the social region.

In [ ]:
def circular_shift_null(ts, n_perm=500, seed=1):
    rng = np.random.default_rng(seed); n, T = ts.shape; null = np.empty(n_perm)
    for j in range(n_perm):
        shifted = np.stack([np.roll(ts[i], rng.integers(1, T)) for i in range(n)])
        null[j] = fisher_mean(leave_one_out_isc(shifted))
    return null

obs = fisher_mean(leave_one_out_isc(data['social/STS']))
null = circular_shift_null(data['social/STS'])
p_perm = (np.sum(null >= obs) + 1) / (len(null) + 1)
print(f'social/STS observed ISC = {obs:.3f}, null mean = {null.mean():.3f}, p = {p_perm:.4f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
names = list(data); means = [fisher_mean(leave_one_out_isc(data[n])) for n in names]
sds = [leave_one_out_isc(data[n]).std() for n in names]
ax[0].bar(names, means, yerr=sds, capsize=4, color=['#2563eb','#0d9488','#94a3b8'])
ax[0].axhline(0, color='k', lw=.8); ax[0].set(ylabel='leave-one-out ISC', title='ISC by region')
ax[1].hist(null, bins=30, color='#94a3b8', alpha=.8, label='null (time-shifted)')
ax[1].axvline(obs, color='#0d9488', lw=2, label='observed (social/STS)')
ax[1].set(xlabel='mean ISC', title='Circular-shift null'); ax[1].legend()
fig.tight_layout(); plt.show()

**Read it.** ISC scales with shared signal (auditory > social > control),
control is correctly ~0, and the real social-region ISC sits far outside the
time-shifted null.

## Part 2 - ISC on real movie-watching fMRI

Same `leave_one_out_isc`, real brains. The ready-to-run path uses **"Partly Cloudy"**
(Richardson et al.) - the Pixar theory-of-mind short - which **nilearn fetches
already preprocessed**:

```bash
pip install "nilearn>=0.10" "matplotlib>=3.8"
```

`code/isc_partly_cloudy.py` fetches the movie data, parcellates the brain with the
Schaefer-2018 atlas, regresses confounds, and computes ISC per region - reusing
`leave_one_out_isc` from Part 1. (`code/isc_naturalistic.py` does the same for
Grand Budapest once you have its derivatives.)

In [ ]:
import sys
from pathlib import Path

# This works from either the repository root or tutorials/naturalistic-isc/.
cwd = Path.cwd()
tutorial_dir = cwd if (cwd / 'code' / 'isc_partly_cloudy.py').exists() else cwd / 'tutorials' / 'naturalistic-isc'
sys.path.insert(0, str(tutorial_dir / 'code'))
import importlib, isc_partly_cloudy as icp
importlib.reload(icp)
icp.main()   # nilearn downloads ~a few hundred MB once; prints top ISC regions + saves a brain map

### What you should find (we executed this on 13 subjects)

ISC mean ~0.13, max ~0.34. The strongest synchronization is in **visual cortex**
(everyone sees the same frames), but it is also high across the **social brain**:
**dorsomedial PFC** (mentalizing) sits near the top alongside vision, and
**temporo-occipital-parietal** cortex near the **TPJ** follows. Partly Cloudy was
built to engage theory of mind, and the synchronized regions show it.

> **Caveat:** ISC needs careful preprocessing/motion control. For real inference
> (subject-level permutation, multiple-comparison correction) use
> [BrainIAK](https://brainiak.org).

### You did it
You measured where brains synchronize to a shared social experience - without
modeling the stimulus - and watched the social brain track a movie full of people.